Separable 4D convolutionn for 2D I/O system

---

Kishore Kumar Tarafdar, Date: 06-06-2025


In [7]:
pwd

'/data1/kishoretarafdar/src.port/VolterraMRAsystems.v0/VolterraSys/ndconvolutions/MakingOfQSI3Dand2Dkernels.pynb'

In [1]:
!python --version

Python 3.12.7


        Disable GPU: Force tensorflow to select CPU

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]="-1"    
import tensorflow as tf

2025-06-30 02:40:04.039584: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1751231404.071088 2280775 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1751231404.080118 2280775 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-30 02:40:04.120440: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


        Select a GPU with a memory limit

In [ ]:
import tensorflow as tf
print(f"TensorFlow version {tf.__version__}")
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
gpus = tf.config.list_physical_devices('GPU')
len(gpus)

Select one GPU

        Restrict code to use a particular GPU...

In [ ]:
# # include ../dirx 
mylibpath = [
    '/home/kishoretarafdar/bin',
    '/data1/kishoretarafdar/src.port/NSLI.v00/utils.Volterra'
    #'/home/k/PLAYGROUND10GB/SKULSTRIPpaper__'
    ]
import sys
[sys.path.insert(0,_) for _ in mylibpath]
del mylibpath

from tf_select_a_gpu import select_a_gpu

In [ ]:
break

In [ ]:
# select_gpu = gpus[gpu_id]
memory_limit = 48#GB
select_a_gpu(gpus, gpu_id=2, memory_limit=memory_limit)
# del gpu_id, select_a_gpu, select_gpu

# Separable conv4d with 2d kernels

        Quadratic NLSI for 2D I/O

    
        Strategy tested with nonseparable conv2d 
        Perfect match with one channel input
        !! Does not match when multiple channel input

        !! Not possible to test the strategy with nonseparable high dimensional convolutions
        (apply update when a libray is located online for nonseparable 4d convolutions)

In [4]:

# from tensorflow.keras.layers import Layer

# # include ../dirx 
mylibpath = [
    '/data1/kishoretarafdar/src.port/VolterraMRAsystems.v0/VolterraSys/ndconvolutions'
    ]
import sys
[sys.path.insert(0,_) for _ in mylibpath]
del mylibpath


import tensorflow as tf
from SeparableConvNDlayout import SeparableConvND


class SeparableConv4D(SeparableConvND):
    """Separable 4D convolution using 2D kernels

    VolterraSys: Multidimensional linear and nonlinear Volterra kernels in natural and multiresolution bases.
    Copyright (C) 2025 Kishore Kumar Tarafdar

    This program is free software: you can redistribute it and/or modify
    it under the terms of the GNU General Public License as published by
    the Free Software Foundation, either version 3 of the License, or
    (at your option) any later version.

    This program is distributed in the hope that it will be useful,
    but WITHOUT ANY WARRANTY; without even the implied warranty of
    MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
    GNU General Public License for more details.

    You should have received a copy of the GNU General Public License
    along with this program.  If not, see <https://www.gnu.org/licenses/>.   
    
    --kkt@29-06-2025"""
    def __init__(self, filters, kernel=None, kernel_size=None, **kwargs):
        super().__init__(filters=filters, kernel=kernel, kernel_size=kernel_size, **kwargs)


    # class SeparableConv4D(Layer):
        
        # def __init__(self, filters, kernel=None, kernel_size=None, **kwargs):
        #     super(SeparableConv4D, self).__init__(**kwargs)
        #     self.filters = filters        
        #     if kernel is None: 
        #         self.kernel_size = kernel_size
        #         self.kernel = kernel 
        #     else: 
        #         self.kernel = kernel
        #         self.kernel_size = tf.shape(kernel)[0]
                    
    # def build(self, input_shape):
    #     self.inchannels = input_shape[-1]
    #     # self.filters = input_shape[-1]
    #     ndim = (len(input_shape) - 2)//2  # exclude batch and channel dims
    #     spatial_shape = input_shape[1:-1]  # [N1, N2, N3, ...]
        
    #     if self.kernel is None:
    #         # Experimental pointwise kernel! a nontrainable pointwise convolution with ones to match 
    #         # input channels to number of output channels (for smooth separable conv with self.kernel)
    #         # Determine shape for pointwise kernel (1x1x1...x1, in_channels, out_channels)
    #         pointwise_shape = (1,) * ndim + (self.inchannels, self.filters) 
    #         print('ps', pointwise_shape)
    #         self.pointwise = self.add_weight(
    #             name='match_inchannels_with_outchannels_number',
    #             # shape=(1, 1, input_shape[-1], self.filters),
    #             shape = pointwise_shape,
    #             initializer='ones',
    #             trainable=False)
    #         # Create a 2D kernel that will be applied to both spatial dimensions
    #         # Shape for separable spatial kernel: (K, K, ..., K, filters, filters)
    #         kernel_shape = (self.kernel_size,) * ndim + (self.filters, self.filters)
    #         self.kernel = self.add_weight(
    #             name='separable_kernel2d',
    #             # shape=(self.kernel_size, self.kernel_size, input_shape[-1], self.filters),
    #             # shape=(self.kernel_size, self.kernel_size, self.filters, self.filters), ## update after pointwise
    #             shape=kernel_shape,
    #             initializer='glorot_uniform',
    #             trainable=True)
        
    def call(self, inputs):
        return self.__separable_conv4d(inputs)
        

    # Function: 4D separable convolution using a single 2D kernel
    def __separable_conv4d(self, x):
        # x: shape [B, N1, N2, N3, N4, C]
        # B, N1, N2, N3, N4, C = x.shape
        # Get static shape for dimensions that shouldn't change
        input_shape = x.shape.as_list()
        N1, N2, N3, N4 = input_shape[1], input_shape[2], input_shape[3], input_shape[4]
        
        # Get dynamic batch size
        B = tf.shape(x)[0]

        # print(' x ', x.shape)
        ## Step 1: Convolve over (N3, N4)
        x1 = tf.reshape(x, [-1, N3, N4, self.inchannels])  # shape: (B*N3*N4, N1, N2, C)
        x1 = tf.nn.convolution(x1, self.pointwise, padding='SAME')
        y1 = tf.nn.convolution(x1, self.kernel, padding='SAME')
        # print(f'+pointwise kernel {self.pointwise.shape} \n+kernel {self.kernel.shape}, \n x1 {x1.shape}, \n y1 { y1.shape}')
        # y1 = tf.reshape(y1, [B, N1, N2, N3, N4, self.inchannels])   # (B, N1, N2, N3, N4, C) ## OK
        y1 = tf.reshape(y1, [B, N1, N2, N3, N4, self.filters])   # (B, N1, N2, N3, N4, C) 
        # print(' y1 ', y1.shape)
        y1 = tf.transpose(y1, perm=[0,3,4,1,2,5])           
        # print('Ty1 ', y1.shape)

        ## Step 2: Convolve over (N1, N2)
        # x2 = tf.reshape(y1, [-1, N1, N2, self.inchannels])  # shape: (B*N1*N2, N3, N4, C) ## OK
        x2 = tf.reshape(y1, [-1, N1, N2, self.filters])  # shape: (B*N1*N2, N3, N4, C) ## OK
        # print(' x2 or reshape y1 ', x2.shape)
        y2 = tf.nn.convolution(x2, self.kernel, padding='SAME')
        # print(' y2 ', y2.shape)
        # y2 = tf.reshape(y2, [B, N1, N2, N3, N4, self.inchannels])    # final shape ## OK
        y2 = tf.reshape(y2, [B, N3, N4, N1, N2,  self.filters])    # final shape
        # print('+y2 ', y2.shape)
        
        y2 = tf.transpose(y2, perm=[0,3,4,1,2,5])
        # print('Ty2 ', y2.shape)
        return y2
        
    # def get_kernel(self):
    #     """Returns the kernel weights as a numpy array"""
    #     return self.kernel.numpy()

    # # def compute_output_shape(self, input_shape):
    # #     return (input_shape[0], input_shape[1], input_shape[2], input_shape[3], input_shape[4], self.filters)
    
    # def get_config(self):
    #     config = super().get_config()
    #     config.update({
    #         'filters': self.filters,
    #         'kernel_size': self.kernel_size
    #     })
    #     return config

if __name__=='__main__':
    # Create a model using the layer
    inputs = tf.keras.Input(shape=(16, 16, 16, 16, 2))  # (N1, N2, N3, N4, C)
    inputs = tf.keras.Input(shape=(128, 128, 64, 64, 2))  # (N1, N2, N3, N4, C)
    inputs = tf.keras.Input(shape=(64, 64, 128, 128, 2))  # (N1, N2, N3, N4, C)
    outputs = SeparableConv4D(filters=13, kernel_size=3)(inputs)
    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    # model.build(None)
    model.summary()

    # # # Test with random data
    # test_input = tf.random.normal([2, 16, 16, 16, 16, 2])
    # output = model(test_input)
    # print(output.shape)
    # del outputs, model, inputs, test_input, output


    ## Example2
    # Input dimensions
    B, N1, N2, N3, N4, C = 2, 8, 8, 8, 8, 3  # Batch size, 4D volume size, channels
    # B, N1, N2, N3, N4, C = 1, 2, 2, 2, 2, 1  # Batch size, 4D volume size, channels
    # B, N1, N2, N3, N4, C = 1, 3, 3, 2, 2, 1  # Batch size, 4D volume size, channels
    x = tf.random.normal((B, N1, N2, N3, N4, C))
    x.shape

    # layer = SeparableConv4D(filters=x.shape[-1], kernel_size=3) ## OK
    layer = SeparableConv4D(filters=13, kernel_size=3)
    yout = layer(x)
    kernel2d = layer.get_kernel()
    print("Kernel shape:", kernel2d.shape)  # Should be (3, 3, 2, 32)
    yout.shape


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_8 (InputLayer)      │ (None, 64, 64, 128,    │             0 │
│                                 │ 128, 2)                │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv4d_4              │ (None, 64, 64, 128,    │         1,547 │
│ (SeparableConv4D)               │ 128, 13)               │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,547 (6.04 KB)

 Trainable params: 1,521 (5.94 KB)

 Non-trainable params: 26 (104.00 B)

Kernel shape: (3, 3, 13, 13)


In [ ]:
# # Input dimensions
# B, N1, N2, N3, N4, C = 2, 8, 8, 8, 8, 3  # Batch size, 4D volume size, channels
# # B, N1, N2, N3, N4, C = 1, 2, 2, 2, 2, 1  # Batch size, 4D volume size, channels
# # B, N1, N2, N3, N4, C = 1, 3, 3, 2, 2, 1  # Batch size, 4D volume size, channels
# x = tf.random.normal((B, N1, N2, N3, N4, C))
# x.shape


In [ ]:
# # layer = SeparableConv4D(filters=x.shape[-1], kernel_size=3) ## OK
# layer = SeparableConv4D(filters=13, kernel_size=3)
# yout = layer(x)
# kernel2d = layer.get_kernel()
# print("Kernel shape:", kernel2d.shape)  # Should be (3, 3, 2, 32)
# yout.shape

In [ ]:
break

In [ ]:
# # Shared 2D convolution kernel (C -> C to keep channels same)
# kH, kW = 3, 3
# kH, kW = 2, 2
# kernel2d = tf.random.normal((kH, kW, C, C))  # single 2D kernel reused



# Run the separable 4D convolution
y = separable_conv4d(x, kernel2d)

# Print shapes
print("Input shape :", x.shape)
print("Kernel 2D shape:", kernel2d.shape)
print("Output shape:", y.shape)


In [ ]:
# # Compare
diff = tf.reduce_max(tf.abs(y - yout))
print("Max absolute difference:", diff.numpy())
print("Outputs match:", tf.reduce_all(tf.abs(y - yout) < 1e-4).numpy())

        CHECK PASS: both layer and function giving same outpput

In [ ]:
## 
break

### separable_conv4d main

In [ ]:
import tensorflow as tf

# Function: 4D separable convolution using a single 2D kernel
def separable_conv4d(x, kernel2d):
    # x: shape [B, N1, N2, N3, N4, C]
    B, N1, N2, N3, N4, C = x.shape

    ## Step 1: Convolve over (N1, N2)
    x1 = tf.reshape(x, [B * N3 * N4, N1, N2, C])  # shape: (B*N3*N4, N1, N2, C)
    y1 = tf.nn.convolution(x1, kernel2d, padding='SAME')
    y1 = tf.reshape(y1, [B, N1, N2, N3, N4, C])   # (B, N1, N2, N3, N4, C)
    y1 = tf.transpose(y1, perm=[0,3,4,1,2,5])
   
    
    # print('+y1 ', y1.shape)


    ## Step 2: Convolve over (N3, N4)
    x2 = tf.reshape(y1, [B * N1 * N2, N3, N4, C])  # shape: (B*N1*N2, N3, N4, C)
    y2 = tf.nn.convolution(x2, kernel2d, padding='SAME')
    y2 = tf.reshape(y2, [B, N1, N2, N3, N4, C])    # final shape
    y2 = tf.transpose(y2, perm=[0,3,4,1,2,5])

    return y2

# -------------------------------
# Test the function with dummy data
# -------------------------------

# Input dimensions
B, N1, N2, N3, N4, C = 2, 8, 8, 8, 8, 3  # Batch size, 4D volume size, channels
B, N1, N2, N3, N4, C = 1, 2, 2, 2, 2, 1  # Batch size, 4D volume size, channels
# B, N1, N2, N3, N4, C = 1, 3, 3, 2, 2, 1  # Batch size, 4D volume size, channels
x = tf.random.normal((B, N1, N2, N3, N4, C))

# Shared 2D convolution kernel (C -> C to keep channels same)
kH, kW = 3, 3
kH, kW = 2, 2
kernel2d = tf.random.normal((kH, kW, C, C))  # single 2D kernel reused

# Run the separable 4D convolution
y = separable_conv4d(x, kernel2d)

# Print shapes
print("Input shape :", x.shape)
print("Kernel 2D shape:", kernel2d.shape)
print("Output shape:", y.shape)



In [ ]:
B, N1, N2, N3, N4, C = x.shape

_ = tf.reshape(x, [B * N3 * N4, N1, N2, C])  # shape: (B*N3*N4, N1, N2, C)
# print(f'x {x.shape} \nx1 ', x1.shape)
# y1 = tf.nn.convolution(x1, kernel2d, padding='SAME')
# print('y1 ', y1.shape)
_r = tf.reshape(_, [B, N1, N2, N3, N4, C]) 
x.shape, _r.shape


In [ ]:

# # Compare
diff = tf.reduce_max(tf.abs(x - _r))
print("Max absolute difference:", diff.numpy())
print("Outputs match:", tf.reduce_all(tf.abs(x - _r) < 1e-4).numpy())


In [ ]:
## Step 2: Convolve over (N3, N4)
__ = tf.reshape(x, [B * N1 * N2, N3, N4, C])  # shape: (B*N1*N2, N3, N4, C)
# y2 = tf.nn.convolution(x2, kernel2d, padding='SAME')
__r = tf.reshape(__, [B, N1, N2, N3, N4, C])    # final shape
x.shape, __r.shape

In [ ]:
# # Compare
diff = tf.reduce_max(tf.abs(x - __r))
print("Max absolute difference:", diff.numpy())
print("Outputs match:", tf.reduce_all(tf.abs(x - __r) < 1e-4).numpy())

In [ ]:
# print(f'x {x.shape} \nx1 ', x1.shape)
# y1 = tf.nn.convolution(x1, kernel2d, padding='SAME')
# print('y1 ', y1.shape)
_r = tf.reshape(_, [B, N3, N4, N1, N2, C]) 

    alternate method check

In [ ]:
kernel4d = np.einsum('ijco,klco->ijklco', kernel2d, kernel2d)
kernel4d.shape

In [ ]:
# Compute brute-force
# y_ref = tf.einsum('bijklc,ijklco->bijklo', x, kernel4d)



In [ ]:
# Compute fast method
# y_fast = separable_4d_conv(x, kernel2d)

# # Compare
# diff = tf.reduce_max(tf.abs(y_ref - y_fast))
# print("Max absolute difference:", diff.numpy())
# print("Outputs match:", tf.reduce_all(tf.abs(y_ref - y_fast) < 1e-4).numpy())


In [ ]:
# np.squeeze(y_ref)

In [ ]:
# y_fast

In [ ]:
# tf.einsum('bijklc,ijklco->bijklo', x, kernel4d)

In [ ]:
# for b in range(x.shape[0]):
#   for n1 in range(x.shape[1]):
#     for n2 in range(x.shape[2]):
#       for n3 in range(x.shape[3]):
#         for n4 in range(x.shape[4]):
#           y[b,n1,n2,n3,n4,c], 

    Try conv 6d with 2d separable kernel

In [ ]:
import tensorflow as tf

def separable_6d_conv(x, kernel2d):
    # x: [B, N1, N2, N3, N4, N5, N6, C]
    B, N1, N2, N3, N4, N5, N6, C = x.shape
    O = kernel2d.shape[-1]

    # Step 1: Convolve over (N1, N2)
    x1 = tf.reshape(x, [B * N3 * N4 * N5 * N6, N1, N2, C])
    y1 = tf.nn.convolution(x1, kernel2d, padding='SAME')
    print(y1.shape)
    y1 = tf.reshape(y1, [B, N1, N2, N3, N4, N5, N6, C])
    # y1 = tf.reshape(y1, [B, N3, N4, N5, N6, N1, N2, C])
    # y1 = tf.transpose(y1, [0, 5, 6, 1, 2, 3, 4, 7])  # [B, N1, N2, N3, N4, N5, N6, C]
    # y1 = tf.transpose(y1, perm=[0,4,5,6,1,2,3,7])
    # y1 = tf.transpose(y1, perm=[0,1,2,3,4,5,6,7])

    # Step 2: Convolve over (N3, N4)
    x2 = tf.reshape(y1, [B * N1 * N2 * N5 * N6, N3, N4, C])
    y2 = tf.nn.convolution(x2, kernel2d, padding='SAME')
    y2 = tf.reshape(y2, [B, N1, N2, N3, N4, N5, N6, C])
    # y2 = tf.reshape(y2, [B, N1, N2, N5, N6, N3, N4, C])
    y2 = tf.transpose(y2, [0, 1, 2, 5, 6, 3, 4, 7])  # [B, N1, N2, N3, N4, N5, N6, C]

    # Step 3: Convolve over (N5, N6)
    x3 = tf.reshape(y2, [B * N1 * N2 * N3 * N4, N5, N6, C])
    y3 = tf.nn.convolution(x3, kernel2d, padding='SAME')
    y3 = tf.reshape(y3, [B, N1, N2, N3, N4, N5, N6, C])

    return y3

# Input shape
B, N1, N2, N3, N4, N5, N6, C = 1, 2, 2, 2, 2, 2, 2, 1
x = tf.random.normal((B, N1, N2, N3, N4, N5, N6, C))
# O = 2

# Shared kernel
kH, kW = 2, 2
kernel2d = tf.random.normal((kH, kW, C, C))

# Apply 6D separable conv
y = separable_6d_conv(x, kernel2d)

# Output
print("Input shape :", x.shape)
print("Kernel shape:", kernel2d.shape)
print("Output shape:", y.shape)
